# API contract tests (live, M2M)

Session 35 (build 4, wave 4-3). Calls the deployed Semantic Ground Truth app as the M2M service principal and checks every REST route and MCP tool in `tests/contract/manifest.json` (generated from `shared/contracts/schemas.ts`, same source as `shared/contracts/generated/contracts.schema.json`).

* **Seeds its own data:** three run-scoped assets `contract_test_<run_id>_m1..m3` and a campaign `contract-test-<run_id>`, then votes, skips, synonyms and examples through the API.
* **Asserts** the success status, JSON Schema validity of the body (draft 2020-12, closed objects) and exact seeded values; negative cases the admin SPN can exercise (400, 404, some 409); error bodies must match `ErrorResponse` with the right `error.code`.
* **Records `skipped_live`** (with a reason) for statuses that cannot be produced live: 401 (the Apps proxy redirects before the app), 403 (the SPN is an app admin; covered in-process), 500/502 (needs a broken data layer or upstream), and happy paths that would create real, undeletable resources.
* **MCP:** `initialize`, `tools/list` (13 tools, `_meta.ui.resourceUri` on the card tools), `resources/read` (MCP Apps MIME type, non-empty HTML) and `tools/call` per tool.
* **Cleans up:** archives the campaigns, removes the test group member, restores settings. There is no delete API for assets, so the seeded assets stay (unique names, domain `ContractTest`).
* **Writes** one row per check to `<catalog>.<schema>.api_contract_results` and **fails** if any check failed or any manifest entry has no row.

The in-process suite (`tests/contract/*.contract.test.ts`, vitest) covers the same contracts against a seeded fake data layer on every Apps build.

In [0]:
%pip install --upgrade databricks-sdk jsonschema requests
dbutils.library.restartPython()

In [0]:
import re
import time

dbutils.widgets.text("app_name", "dev-semantic-ground-truth", "App name")
dbutils.widgets.text("secret_scope", "semantic_ground_truth_credentials", "Secret scope")
dbutils.widgets.text("client_id_key", "m2m_client_id_dev", "M2M client_id key")
dbutils.widgets.text("client_secret_key", "m2m_client_secret_dev", "M2M client_secret key")
dbutils.widgets.text("catalog", "hls_fde_dev", "UC catalog")
dbutils.widgets.text("schema", "dev_matthew_giglia_semantic_ground_truth", "UC schema")
dbutils.widgets.text("run_id", "", "Job run id (empty = manual run)")

APP_NAME = dbutils.widgets.get("app_name").strip()
SECRET_SCOPE = dbutils.widgets.get("secret_scope").strip()
CLIENT_ID_KEY = dbutils.widgets.get("client_id_key").strip()
CLIENT_SECRET_KEY = dbutils.widgets.get("client_secret_key").strip()
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
RUN_ID = dbutils.widgets.get("run_id").strip() or f"manual-{int(time.time())}"

# Identifiers are interpolated into SQL below: allow only plain UC names.
for _label, _ident in (("catalog", CATALOG), ("schema", SCHEMA)):
    if not re.fullmatch(r"[A-Za-z0-9_]+", _ident):
        raise ValueError(f"invalid {_label} name: {_ident!r}")
if not APP_NAME:
    raise ValueError("app_name is required")

RESULTS_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`api_contract_results`"
TAG = "contract_test_" + re.sub(r"[^0-9a-z]", "_", RUN_ID.lower())
CAMPAIGN_NAME = f"contract-test-{RUN_ID}"
UNKNOWN = "00000000-0000-4000-8000-000000000000"
print(f"app={APP_NAME} run_id={RUN_ID} results={RESULTS_TABLE}")

In [0]:
import json
from pathlib import Path

from jsonschema import Draft202012Validator

# This notebook lives at <app root>/src/notebooks/; the bundle syncs the whole app.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_nb = Path(_nb_path if _nb_path.startswith("/Workspace") else "/Workspace" + _nb_path)
APP_ROOT = _nb.parents[2]

MANIFEST = json.loads((APP_ROOT / "tests" / "contract" / "manifest.json").read_text())
SCHEMA_DOC = json.loads((APP_ROOT / MANIFEST["schemaFile"]).read_text())
DEFS = SCHEMA_DOC["$defs"]
ROUTES = {r["key"]: r for r in MANIFEST["routes"]}
TOOLS = {t["name"]: t for t in MANIFEST["mcpTools"]}
RESOURCES = {r["uri"]: r for r in MANIFEST["mcpResources"]}

_validators = {}


def schema_errors(ref, instance):
    """JSON Schema errors of `instance` against a manifest $ref (empty list = valid)."""
    if ref is None:
        return []
    key = ref["$ref"]
    if key not in _validators:
        _validators[key] = Draft202012Validator({"$schema": SCHEMA_DOC["$schema"], "$ref": key, "$defs": DEFS})
    errs = sorted(_validators[key].iter_errors(instance), key=lambda e: list(e.absolute_path))
    return [f"{'/'.join(map(str, e.absolute_path)) or '<root>'}: {e.message[:200]}" for e in errs[:5]]


print(f"{len(ROUTES)} routes, {len(TOOLS)} MCP tools, {len(RESOURCES)} MCP resources from {APP_ROOT}")

In [0]:
from datetime import datetime, timezone
from urllib.parse import quote

import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
_url = w.apps.get(APP_NAME).url or ""
APP_URL = (_url if _url.startswith("https://") else f"https://{_url}").rstrip("/")

# M2M OAuth (client credentials) as the app's admin SPN; a plain PAT gets 302/403 from the Apps proxy.
m2m_w = WorkspaceClient(
    host=w.config.host,
    client_id=dbutils.secrets.get(scope=SECRET_SCOPE, key=CLIENT_ID_KEY),
    client_secret=dbutils.secrets.get(scope=SECRET_SCOPE, key=CLIENT_SECRET_KEY),
)
session = requests.Session()

RESULTS = []
ERROR_CODE_FOR_STATUS = {400: "BAD_REQUEST", 401: "UNAUTHENTICATED", 403: "FORBIDDEN", 404: "NOT_FOUND",
                         409: "CONFLICT", 500: "INTERNAL", 502: "UPSTREAM_ERROR"}


def _s(v, limit=4000):
    s = v if isinstance(v, str) else json.dumps(v, default=str, sort_keys=True)
    return s if len(s) <= limit else s[:limit] + "...[truncated]"


def record(check_id, kind, method, target, expected, actual, status, duration_ms=0.0):
    RESULTS.append({
        "run_id": RUN_ID, "check_id": check_id, "kind": kind, "method": method, "path_or_tool": target,
        "expected": _s(expected), "actual": _s(actual), "status": status,
        "duration_ms": int(duration_ms), "ts": datetime.now(timezone.utc),
    })
    mark = {"pass": "PASS", "fail": "FAIL", "skipped_live": "SKIP"}[status]
    print(f"[{mark}] {check_id}" + ("" if status == "pass" else f" -> {_s(actual, 400)}"))


def http(method, path, body=None, query=None, headers=None):
    t0 = time.monotonic()
    r = session.request(method, APP_URL + path, json=body, params=query, timeout=120, allow_redirects=False,
                        headers={**m2m_w.config.authenticate(), **(headers or {})})
    ms = (time.monotonic() - t0) * 1000
    try:
        data = r.json()
    except ValueError:
        data = r.text[:2000]
    return r.status_code, data, ms, r


def fill(path, params):
    return re.sub(r":(\w+)", lambda m: quote(str(params[m.group(1)]), safe=""), path)


def subset_problems(expected, actual, where=""):
    """Every key in `expected` is in `actual` with an equal value (recursive for objects)."""
    if isinstance(expected, dict):
        if not isinstance(actual, dict):
            return [f"{where or '<root>'}: expected object, got {type(actual).__name__}"]
        out = []
        for k, v in expected.items():
            out += [f"{where}.{k}: missing"] if k not in actual else subset_problems(v, actual[k], f"{where}.{k}")
        return out
    return [] if expected == actual else [f"{where or '<root>'}: expected {expected!r}, got {_s(actual, 200)}"]


def has(pred, msg):
    """expect= helper: one problem `msg` unless pred(body) is truthy."""
    def _check(body):
        try:
            return [] if pred(body) else [msg]
        except Exception as e:  # a malformed body is a contract failure, not a notebook crash
            return [f"{msg} ({type(e).__name__}: {e})"]
    return _check


def check_route(key, label="happy", *, params=None, body=None, query=None, status=None, expect=None):
    """Call one manifest route; check status, schema, error code and expected values. Returns the body on the expected status."""
    r = ROUTES[key]
    want = status or r["status"]
    is_error = want != r["status"]
    check_id = f"route:{key}:{label}"
    try:
        code, data, ms, resp = http(r["method"], fill(r["path"], params or {}), body, query)
    except Exception as e:
        record(check_id, "route", r["method"], key, {"status": want}, f"request failed: {type(e).__name__}: {e}", "fail")
        return None
    problems = []
    if code in (301, 302, 303, 307):
        problems.append(f"redirect {code} to {resp.headers.get('location', '')[:120]}: SPN lacks CAN_USE or token is not OAuth")
    elif code != want:
        problems.append(f"status {code} != {want}")
    else:
        problems += schema_errors(MANIFEST["errorResponse"] if is_error else r["response"], data)
        if is_error:
            got = (data.get("error") or {}).get("code") if isinstance(data, dict) else None
            if got != ERROR_CODE_FOR_STATUS.get(want):
                problems.append(f"error.code {got!r} != {ERROR_CODE_FOR_STATUS.get(want)!r}")
        elif expect is not None and not problems:
            try:
                problems += expect(data) if callable(expect) else subset_problems(expect, data)
            except Exception as e:  # malformed body: a contract failure, not a notebook crash
                problems.append(f"expectation raised {type(e).__name__}: {e}")
    record(check_id, "route", r["method"], key,
           {"status": want, "values": "<predicate>" if callable(expect) else expect},
           {"status": code, "problems": problems, "body": data}, "fail" if problems else "pass", ms)
    return data if code == want else None


def skip_live(key, label, reason):
    record(f"route:{key}:{label}", "route", ROUTES[key]["method"], key, label, reason, "skipped_live")


def invariant(check_id, ok, expected, actual):
    record(f"invariant:{check_id}", "invariant", "", check_id, expected, actual, "pass" if ok else "fail")


_t = time.monotonic()
_code, _, _, _resp = http("GET", "/health")
if _code in (301, 302, 303, 307):
    raise RuntimeError(f"GET /health -> {_code}: the M2M SPN lacks CAN_USE on app {APP_NAME}")
print(f"{APP_URL} auth={m2m_w.config.auth_type} health={_code}")

In [0]:
SEED = {}
NOT_EXECUTABLE = "seeded contract-test assets carry no executable SQL; covered in-process (fake upstream)"
UNDELETABLE = "happy path creates a real resource that no API can delete; covered in-process"

# -- identity / config --
check_route("GET /health", expect={"status": "ok", "db": True})
check_route("GET /api/config", expect=has(lambda d: d["executive_dashboard_url"].startswith("https://"), "executive_dashboard_url is not https"))
ME = check_route("GET /api/me", expect=has(lambda d: "admin" in d["roles"], "M2M SPN is not an app admin")) or {}
ME_ID = ME.get("userId")

# -- seed three assets --
for i in (1, 2, 3):
    name = f"{TAG}_m{i}"
    yaml_content = f"name: {name}\n"
    d = check_route("POST /api/assets/register", f"happy_m{i}", body={
        "assetType": "measure", "name": name, "yamlContent": yaml_content, "environment": "dev",
        "targetCatalog": CATALOG, "targetSchema": SCHEMA, "domain": "ContractTest",
    }, expect={"created": True, "asset": {"name": name, "displayName": name, "assetType": "measure", "yamlContent": yaml_content,
                                           "environment": "dev", "targetCatalog": CATALOG, "targetSchema": SCHEMA, "domain": "ContractTest"}})
    if d:
        SEED[f"a{i}"] = d["asset"]["id"]
check_route("POST /api/assets/register", "error_400", body={"assetType": "measure", "name": ""}, status=400)
A1, A2, A3 = SEED.get("a1", UNKNOWN), SEED.get("a2", UNKNOWN), SEED.get("a3", UNKNOWN)

# -- assets --
check_route("GET /api/assets/:assetId", params={"assetId": A1}, expect={"id": A1, "name": f"{TAG}_m1", "status": "PENDING"})
check_route("GET /api/assets/:assetId", "error_400", params={"assetId": "not-a-uuid"}, status=400)
check_route("GET /api/assets/:assetId", "error_404", params={"assetId": UNKNOWN}, status=404)
check_route("GET /api/assets/:assetId/versions", params={"assetId": A1}, expect=has(lambda d: all(v["assetId"] == A1 for v in d), "versions for another asset"))
check_route("GET /api/assets/:assetId/versions", "error_400", params={"assetId": "not-a-uuid"}, status=400)
check_route("GET /api/assets/:assetId/versions", "error_404", params={"assetId": UNKNOWN}, status=404)
check_route("GET /api/assets", query={"limit": 200}, expect={"limit": 200, "offset": 0})
check_route("GET /api/assets", "error_400", query={"limit": 0}, status=400)

# -- campaign scoped to the seeded assets --
scope = sorted(SEED.values())
c = check_route("POST /api/campaigns", body={"name": CAMPAIGN_NAME, "campaignType": "manual", "scopeAssetIds": scope},
                expect=lambda d: subset_problems({"name": CAMPAIGN_NAME, "campaignType": "manual", "status": "active"}, d)
                + ([] if sorted(d["scopeAssetIds"]) == scope else [f"scopeAssetIds {d['scopeAssetIds']} != {scope}"]))
C = (c or {}).get("id", UNKNOWN)
check_route("POST /api/campaigns", "error_400", body={"name": "", "campaignType": "manual"}, status=400)
check_route("GET /api/campaigns", query={"status": "active"}, expect=has(lambda d: any(x["id"] == C for x in d), "seeded campaign not listed"))
check_route("GET /api/campaigns", "error_400", query={"status": "bogus"}, status=400)
check_route("GET /api/campaigns/:campaignId", params={"campaignId": C}, expect={"id": C, "name": CAMPAIGN_NAME, "status": "active"})
check_route("GET /api/campaigns/:campaignId", "error_400", params={"campaignId": "not-a-uuid"}, status=400)
check_route("GET /api/campaigns/:campaignId", "error_404", params={"campaignId": UNKNOWN}, status=404)
check_route("PATCH /api/campaigns/:campaignId/status", params={"campaignId": C}, body={"status": "paused"}, expect={"id": C, "status": "paused"})
check_route("PATCH /api/campaigns/:campaignId/status", "happy_reactivate", params={"campaignId": C}, body={"status": "active"}, expect={"id": C, "status": "active"})
check_route("PATCH /api/campaigns/:campaignId/status", "error_400", params={"campaignId": C}, body={"status": "done"}, status=400)
check_route("PATCH /api/campaigns/:campaignId/status", "error_404", params={"campaignId": UNKNOWN}, body={"status": "paused"}, status=404)
skip_live("POST /api/campaigns/freshness-check", "happy", "creates a real freshness campaign over every workspace asset; covered in-process")

# -- review loop --
check_route("GET /api/review/next", query={"campaignId": C},
            expect=has(lambda d: d is not None and d["asset"]["id"] in SEED.values(), "next card is not a seeded asset"))
check_route("GET /api/review/next", "error_400", query={"assetType": "chart"}, status=400)
check_route("GET /api/review/next", "error_404", query={"campaignId": UNKNOWN}, status=404)
stats0 = check_route("GET /api/review/stats/me") or {}

check_route("POST /api/review/vote", body={"assetId": A1, "voteType": "approve"},
            expect={"vote": {"assetId": A1, "voteType": "approve", "source": "web", **({"userId": ME_ID} if ME_ID else {})}})
check_route("POST /api/review/vote", "error_400", body={"assetId": A1, "voteType": "skip"}, status=400)
check_route("POST /api/review/vote", "error_404", body={"assetId": UNKNOWN, "voteType": "approve"}, status=404)

SYN = f"{TAG} synonym"
check_route("POST /api/review/synonym", body={"assetId": A1, "action": "add", "synonym": SYN},
            expect=has(lambda d: any(s["synonym"] == SYN and s["assetId"] == A1 and s["approvals"] == 0 for s in d), f"synonym {SYN!r} not returned"))
check_route("POST /api/review/synonym", "error_400", body={"assetId": A1, "action": "add", "synonym": ""}, status=400)
check_route("POST /api/review/synonym", "error_404", body={"assetId": UNKNOWN, "action": "add", "synonym": SYN}, status=404)

Q = f"How many {TAG} rows are there?"
check_route("POST /api/review/example", body={"assetId": A1, "action": "add", "questionText": Q},
            expect=has(lambda d: any(e["questionText"] == Q and e["assetId"] == A1 and e["approved"] is False for e in d), "example not returned"))
check_route("POST /api/review/example", "error_400", body={"assetId": A1, "action": "flag"}, status=400)
check_route("POST /api/review/example", "error_404", body={"assetId": UNKNOWN, "action": "add", "questionText": Q}, status=404)

conf_a2 = (check_route("POST /api/admin/recompute/:assetId", "happy_before_skip", params={"assetId": A2}) or {})
sk = check_route("POST /api/review/skip", body={"assetId": A2, "campaignId": C, "reason": "need_context"},
                 expect={"skip": {"assetId": A2, "campaignId": C, "reason": "need_context", "source": "web"}})
check_route("POST /api/review/skip", "error_400", body={"assetId": A2, "reason": "bored"}, status=400)
check_route("POST /api/review/skip", "error_404", body={"assetId": UNKNOWN}, status=404)
stats1 = check_route("GET /api/review/stats/me", "happy_after") or {}
if stats0 and stats1:
    _sk0, _sk1 = stats0.get("skippedCount", -1), stats1.get("skippedCount")
    _tv0, _tv1 = stats0.get("totalVotes", -1), stats1.get("totalVotes")
    invariant("skip increments skippedCount by 1", _sk1 == _sk0 + 1, _sk0 + 1, _sk1)
    invariant("skip is not a vote (totalVotes +1 from the single approve)", _tv1 == _tv0 + 1, _tv0 + 1, _tv1)
conf_a2_after = check_route("POST /api/admin/recompute/:assetId", "happy_after_skip", params={"assetId": A2}) or {}
if conf_a2 and conf_a2_after:
    keep = ("totalVotes", "approvals", "rejections", "edits", "confidence")
    invariant("skip leaves confidence and vote count unchanged", all(conf_a2.get(k) == conf_a2_after.get(k) for k in keep),
              {k: conf_a2.get(k) for k in keep}, {k: conf_a2_after.get(k) for k in keep})
nxt = check_route("GET /api/review/next", "happy_after_skip", query={"campaignId": C})
_next_id = ((nxt or {}).get("asset") or {}).get("id")
invariant("cooldown hides the skipped asset", _next_id != A2, f"next != {A2}", _next_id)

skip_live("POST /api/review/run-query", "happy", NOT_EXECUTABLE)
check_route("POST /api/review/run-query", "error_400", body={}, status=400)
check_route("POST /api/review/run-query", "error_404", body={"assetId": UNKNOWN}, status=404)
check_route("POST /api/admin/recompute/:assetId", params={"assetId": A1}, expect={"assetId": A1, "totalVotes": 1, "approvals": 1, "rejections": 0, "edits": 0})
check_route("POST /api/admin/recompute/:assetId", "error_400", params={"assetId": "not-a-uuid"}, status=400)
check_route("POST /api/admin/recompute/:assetId", "error_404", params={"assetId": UNKNOWN}, status=404)

check_route("GET /api/leaderboard", query={"period": "all"})
check_route("GET /api/leaderboard", "error_400", query={"period": "year"}, status=400)

# -- sources / import / Unity Catalog browse --
check_route("GET /api/sources")
skip_live("POST /api/sources", "happy", UNDELETABLE)
check_route("POST /api/sources", "error_400", body={}, status=400)
skip_live("POST /api/sources/:id/scan", "happy", "needs a registered source; POST /api/sources happy is not run live")
check_route("POST /api/sources/:id/scan", "error_400", params={"id": "not-a-uuid"}, status=400)
check_route("POST /api/sources/:id/scan", "error_404", params={"id": UNKNOWN}, status=404)
skip_live("POST /api/assets/import", "happy", UNDELETABLE)
check_route("POST /api/assets/import", "error_400", body={"metricViews": "not-a-list"}, status=400)
check_route("POST /api/assets/import", "error_404", body={"sourceId": UNKNOWN}, status=404)

check_route("GET /api/catalog/catalogs", expect=has(lambda d: any(x["name"] == CATALOG for x in d), f"catalog {CATALOG} not listed"))
check_route("GET /api/catalog/schemas", query={"catalog": CATALOG}, expect=has(lambda d: any(x["name"] == SCHEMA for x in d), f"schema {SCHEMA} not listed"))
check_route("GET /api/catalog/schemas", "error_400", query={}, status=400)
check_route("GET /api/catalog/schemas", "error_404", query={"catalog": f"{TAG}_missing"}, status=404)
check_route("GET /api/catalog/metric-views", query={"catalog": CATALOG, "schema": SCHEMA})
check_route("GET /api/catalog/metric-views", "error_400", query={"catalog": CATALOG}, status=400)
check_route("GET /api/catalog/metric-views", "error_404", query={"catalog": CATALOG, "schema": f"{TAG}_missing"}, status=404)

mvs = check_route("GET /api/metric-views", query={"catalog": CATALOG, "schema": SCHEMA}) or []
MV = mvs[0].get("fullName") if isinstance(mvs, list) and mvs and isinstance(mvs[0], dict) else None
if MV:
    check_route("GET /api/metric-views/:fullName", params={"fullName": MV}, expect={"fullName": MV})
    check_route("POST /api/metric-views/:fullName/validate", params={"fullName": MV})
else:
    skip_live("GET /api/metric-views/:fullName", "happy", f"no metric view in {CATALOG}.{SCHEMA}")
    skip_live("POST /api/metric-views/:fullName/validate", "happy", f"no metric view in {CATALOG}.{SCHEMA}")
for key in ("GET /api/metric-views/:fullName", "POST /api/metric-views/:fullName/validate"):
    check_route(key, "error_400", params={"fullName": "not_three_part"}, status=400)
    check_route(key, "error_404", params={"fullName": f"{CATALOG}.{SCHEMA}.{TAG}_missing"}, status=404)

# -- notifications (round-trip, then restore) --
check_route("GET /api/notifications", query={"limit": 20})
check_route("GET /api/notifications", "error_400", query={"limit": 0}, status=400)
check_route("GET /api/notifications/unread-count")
check_route("POST /api/notifications/read", body={"ids": "all"})
check_route("GET /api/notifications/unread-count", "happy_after_read_all", expect={"count": 0})
check_route("POST /api/notifications/read", "error_400", body={"ids": []}, status=400)
prefs = check_route("GET /api/notifications/preferences")
if prefs:
    body = [{"eventType": p["eventType"], "channels": p["channels"], "digest": p["digest"]} for p in prefs]
    check_route("PUT /api/notifications/preferences", body={"preferences": body},
                expect=has(lambda d: sorted((p["eventType"], tuple(p["channels"]), p["digest"]) for p in d)
                           == sorted((p["eventType"], tuple(p["channels"]), p["digest"]) for p in body), "preferences did not round-trip"))
else:
    skip_live("PUT /api/notifications/preferences", "happy", "GET preferences returned nothing to round-trip")
check_route("PUT /api/notifications/preferences", "error_400", body={"preferences": [{"eventType": "vote.cast", "channels": ["in_app"], "digest": False}]}, status=400)

# -- user settings (round-trip, then restore) --
settings0 = check_route("GET /api/me/settings")
if isinstance(settings0, dict) and "theme" in settings0:
    theme = "dark" if settings0["theme"] != "dark" else "light"
    put = check_route("PUT /api/me/settings", body={"theme": theme}, expect={**{k: v for k, v in settings0.items() if k != "updatedAt"}, "theme": theme})
    got = check_route("GET /api/me/settings", "happy_after_put")
    invariant("settings PUT then GET round-trip", put is not None and got == put, put, got)
    restore = {k: v for k, v in settings0.items() if k != "updatedAt"}
    check_route("PUT /api/me/settings", "happy_restore", body=restore, expect=restore)
check_route("PUT /api/me/settings", "error_400", body={}, status=400)
check_route("PUT /api/me/settings", "error_404", body={"defaultCampaignId": UNKNOWN}, status=404)

# -- admin --
cfg = check_route("GET /api/admin/config")
if isinstance(cfg, dict) and "skipCooldownHours" in cfg:
    check_route("PUT /api/admin/config", body={"skipCooldownHours": cfg["skipCooldownHours"]}, expect=cfg)
check_route("PUT /api/admin/config", "error_400", body={"wilsonZ": -1}, status=400)
check_route("GET /api/admin/summary")
check_route("GET /api/admin/groups", expect=has(lambda d: {g["name"] for g in d} >= {"semantic-ground-truth-admins", "semantic-ground-truth-stewards", "semantic-ground-truth-users"}, "missing app groups"))
MEMBER = f"{TAG}@example.com"
STEWARDS = "semantic-ground-truth-stewards"
check_route("POST /api/admin/groups/:name/members", params={"name": STEWARDS}, body={"memberId": MEMBER},
            expect=has(lambda d: d["name"] == STEWARDS and any(m["memberId"] == MEMBER for m in d["members"]), "member not added"))
check_route("POST /api/admin/groups/:name/members", "error_400", params={"name": STEWARDS}, body={}, status=400)
check_route("DELETE /api/admin/groups/:name/members/:memberId", params={"name": STEWARDS, "memberId": MEMBER},
            expect=has(lambda d: d["name"] == STEWARDS and all(m["memberId"] != MEMBER for m in d["members"]), "member not removed"))
check_route("DELETE /api/admin/groups/:name/members/:memberId", "error_400", params={"name": "everyone", "memberId": MEMBER}, status=400)

# -- steward triage --
check_route("GET /api/steward/feedback-queue", query={"campaignId": C, "status": "all"},
            expect=has(lambda d: any(i["assetId"] == A1 and i["votes"]["approve"] == 1 for i in d["items"]), "A1 with 1 approve not in queue"))
check_route("GET /api/steward/feedback-queue", "error_400", query={"status": "bogus"}, status=400)
check_route("GET /api/steward/conflicts", query={"campaignId": C}, expect=has(lambda d: all(g["assetId"] in SEED.values() for g in d), "conflict outside the seeded campaign"))
check_route("GET /api/steward/conflicts", "error_400", query={"assetType": "chart"}, status=400)
skip_live("POST /api/steward/conflicts/:assetId/resolve", "happy", "a conflict needs edit votes from two identities; the job has one (SPN); covered in-process")
check_route("POST /api/steward/conflicts/:assetId/resolve", "error_400", params={"assetId": A1}, body={"resolution": "bogus"}, status=400)
check_route("POST /api/steward/conflicts/:assetId/resolve", "error_404", params={"assetId": UNKNOWN}, body={"resolution": "keep_current"}, status=404)
check_route("GET /api/steward/feedback-branches")
check_route("GET /api/steward/feedback-branches", "error_400", query={"status": "bogus"}, status=400)
skip_live("POST /api/steward/feedback-branches", "happy", "creates a real feedback git branch; covered in-process")
check_route("POST /api/steward/feedback-branches", "error_400", body={"branchName": "not-a-feedback-branch"}, status=400)
check_route("POST /api/steward/feedback-branches", "error_404", body={"batchId": UNKNOWN}, status=404)
skip_live("POST /api/steward/feedback-branches/:batchId/status", "happy", "needs a real feedback batch; covered in-process")
check_route("POST /api/steward/feedback-branches/:batchId/status", "error_400", params={"batchId": UNKNOWN}, body={"status": "done"}, status=400)
check_route("POST /api/steward/feedback-branches/:batchId/status", "error_404", params={"batchId": UNKNOWN}, body={"status": "in_review"}, status=404)

print(f"seed: {SEED} campaign={C} metric_view={MV}")

In [0]:
MCP_HEADERS = {"Accept": "application/json, text/event-stream", "Content-Type": "application/json"}
_rpc_ids = iter(range(1, 1_000_000))


def rpc(method, params=None):
    """One JSON-RPC request to the stateless Streamable HTTP endpoint (JSON or SSE response)."""
    msg = {"jsonrpc": "2.0", "id": next(_rpc_ids), "method": method, **({"params": params} if params is not None else {})}
    t0 = time.monotonic()
    r = session.post(APP_URL + "/mcp", json=msg, timeout=120, allow_redirects=False,
                     headers={**m2m_w.config.authenticate(), **MCP_HEADERS})
    ms = (time.monotonic() - t0) * 1000
    data = None
    if "text/event-stream" in r.headers.get("content-type", ""):
        events = [json.loads(line[5:]) for line in r.text.splitlines() if line.startswith("data:") and line[5:].strip()]
        data = next((e for e in events if e.get("id") == msg["id"]), None)
    else:
        try:
            data = r.json()
        except ValueError:
            data = {"raw": r.text[:2000]}
    return r.status_code, data or {}, ms


def mcp_check(check_id, kind, target, expected, fn):
    """Run fn() -> (problems, actual, ms); record pass/fail."""
    try:
        problems, actual, ms = fn()
    except Exception as e:
        problems, actual, ms = [f"{type(e).__name__}: {e}"], None, 0
    record(check_id, kind, "mcp", target, expected, {"problems": problems, "result": actual}, "fail" if problems else "pass", ms)
    return actual if not problems else None


def _initialize():
    code, data, ms = rpc("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                                         "clientInfo": {"name": "api_contract_tests", "version": "1.0.0"}})
    res = data.get("result") or {}
    caps = res.get("capabilities") or {}
    problems = [p for p, bad in ((f"HTTP {code}", code != 200), (f"error {data.get('error')}", "error" in data),
                                 ("no serverInfo.name", not (res.get("serverInfo") or {}).get("name")),
                                 ("no tools capability", "tools" not in caps), ("no resources capability", "resources" not in caps)) if bad]
    return problems, res, ms


def _tools_list():
    code, data, ms = rpc("tools/list", {})
    tools = (data.get("result") or {}).get("tools") or []
    names = sorted(t["name"] for t in tools)
    problems = [] if code == 200 else [f"HTTP {code}: {data.get('error')}"]
    if names != sorted(TOOLS):
        problems.append(f"tools {names} != manifest {sorted(TOOLS)}")
    for t in tools:
        uri = ((t.get("_meta") or {}).get("ui") or {}).get("resourceUri")
        want = (TOOLS.get(t["name"]) or {}).get("uiResourceUri")
        if uri != want:
            problems.append(f"{t['name']}: _meta.ui.resourceUri {uri!r} != {want!r}")
    return problems, {"count": len(tools), "names": names}, ms


mcp_check("mcp:initialize", "tool", "initialize", "serverInfo + tools/resources capabilities", _initialize)
mcp_check("mcp:tools/list", "tool", "tools/list", {"count": len(TOOLS), "names": sorted(TOOLS)}, _tools_list)

for uri, res_meta in RESOURCES.items():
    def _read(uri=uri, res_meta=res_meta):
        code, data, ms = rpc("resources/read", {"uri": uri})
        contents = (data.get("result") or {}).get("contents") or []
        problems = [] if code == 200 and contents else [f"HTTP {code}: {data.get('error')}"]
        if contents:
            c0, html = contents[0], str(contents[0].get("text") or "")
            if c0.get("mimeType") != res_meta["mimeType"]:
                problems.append(f"mimeType {c0.get('mimeType')!r} != {res_meta['mimeType']!r}")
            if len(html) <= 1000 or "<html" not in html.lower():
                problems.append(f"not an HTML document ({len(html)} chars)")
        return problems, {"mimeType": contents[0].get("mimeType") if contents else None,
                          "chars": len(str(contents[0].get("text") or "")) if contents else 0}, ms
    mcp_check(f"resource:{uri}:read", "resource", uri, res_meta, _read)


def _text(res):
    return "".join(c.get("text", "") for c in res.get("content") or [] if c.get("type") == "text")


def check_tool(name, args, label="happy", expect=None, error_code=None):
    """tools/call: happy = structuredContent valid + expected values; error_code = isError with `Error [CODE]: ` ('*' = any isError)."""
    t = TOOLS[name]

    def _call():
        code, data, ms = rpc("tools/call", {"name": name, "arguments": args})
        res = data.get("result")
        if code != 200 or res is None:
            return [f"HTTP {code}, JSON-RPC error {_s(data.get('error'), 300)}"], data, ms
        text, sc = _text(res), res.get("structuredContent")
        if error_code:
            problems = [] if res.get("isError") else ["expected isError"]
            if error_code != "*" and not text.startswith(f"Error [{error_code}]: "):
                problems.append(f"text {text[:160]!r} does not start with 'Error [{error_code}]: '")
            return problems, {"isError": res.get("isError"), "text": text[:500]}, ms
        if res.get("isError"):
            return [f"isError: {text[:300]}"], res, ms
        if sc is None:
            ok_empty = t.get("emptyText") is not None and text == t["emptyText"]
            return ([] if ok_empty and expect == "empty" else [f"no structuredContent (text {text[:160]!r})"]), {"text": text}, ms
        body = dict(sc) if isinstance(sc, dict) else sc
        problems = []
        if name == "review_skip_card" and isinstance(body, dict):  # documented extra field: next card
            problems += schema_errors({"$ref": "#/$defs/res_GET_api_review_next"}, body.pop("next", None))
        problems += schema_errors(t["output"], body)
        if expect not in (None, "empty") and not problems:
            try:
                problems += expect(sc) if callable(expect) else subset_problems(expect, sc)
            except Exception as e:
                problems.append(f"expectation raised {type(e).__name__}: {e}")
        return problems, sc, ms

    problems_hint = {"knownDefects": t["knownDefects"]} if t.get("knownDefects") else {}
    return mcp_check(f"tool:{name}:{label}", "tool", name,
                     {"args": args, "error_code": error_code, "values": "<predicate>" if callable(expect) else expect, **problems_hint}, _call)


# A3 is the only seeded asset the SPN has neither voted on nor skipped.
check_tool("review_get_next", {"campaign_id": C}, expect=has(lambda d: d["asset"]["id"] == A3, "next card is not A3"))
check_tool("review_get_next", {"asset_type": "chart"}, "invalid_args", error_code="*")
check_tool("review_submit_vote", {"asset_id": A3, "vote_type": "approve"}, expect={"vote": {"assetId": A3, "voteType": "approve", "source": "mcp"}})
check_tool("review_submit_vote", {"asset_id": A3, "vote_type": "skip"}, "invalid_args", error_code="*")
check_tool("review_submit_vote", {"asset_id": UNKNOWN, "vote_type": "approve"}, "error_not_found", error_code="NOT_FOUND")
check_tool("review_add_synonym", {"asset_id": A3, "action": "add", "synonym": f"{SYN} mcp"})
check_tool("review_add_synonym", {"asset_id": A3, "action": "add", "synonym": ""}, "invalid_args", error_code="*")
check_tool("review_add_example", {"asset_id": A3, "action": "add", "question_text": f"{Q} (mcp)"})
check_tool("review_add_example", {"asset_id": A3, "action": "flag"}, "invalid_args", error_code="*")
check_tool("review_get_stats", {})
check_tool("review_get_stats", {"period": "year"}, "invalid_args", error_code="*")
record("tool:review_run_query:happy", "tool", "mcp", "review_run_query", "happy", NOT_EXECUTABLE, "skipped_live")
check_tool("review_run_query", {"asset_id": UNKNOWN}, "error_not_found", error_code="NOT_FOUND")
check_tool("campaign_list", {"status": "active"})
check_tool("campaign_list", {"status": "bogus"}, "invalid_args", error_code="*")
mc = check_tool("campaign_create", {"name": f"{CAMPAIGN_NAME}-mcp", "campaign_type": "manual", "scope_asset_ids": [A3]},
                expect={"name": f"{CAMPAIGN_NAME}-mcp", "campaignType": "manual", "status": "active", "scopeAssetIds": [A3]})
MC = (mc or {}).get("id")
check_tool("campaign_create", {"campaign_type": "manual"}, "invalid_args", error_code="*")
if MC:
    check_tool("campaign_update_status", {"campaign_id": MC, "status": "archived"}, expect={"id": MC, "status": "archived"})
else:
    record("tool:campaign_update_status:happy", "tool", "mcp", "campaign_update_status", "happy", "campaign_create failed: nothing to update", "fail")
check_tool("campaign_update_status", {"campaign_id": UNKNOWN, "status": "done"}, "invalid_args", error_code="*")
check_tool("campaign_update_status", {"campaign_id": UNKNOWN, "status": "paused"}, "error_not_found", error_code="NOT_FOUND")
# Metric view tools run on the caller's OBO token; the Apps proxy may not forward one for an M2M caller.
check_tool("metric_view_list", {"catalog": CATALOG, "schema": SCHEMA})
if MV:
    check_tool("metric_view_get", {"full_name": MV}, expect={"fullName": MV})
    check_tool("metric_view_validate", {"full_name": MV})
else:
    for _n in ("metric_view_get", "metric_view_validate"):
        record(f"tool:{_n}:happy", "tool", "mcp", _n, "happy", f"no metric view in {CATALOG}.{SCHEMA}", "skipped_live")
check_tool("metric_view_get", {}, "invalid_args", error_code="*")
check_tool("metric_view_validate", {}, "invalid_args", error_code="*")
check_tool("review_skip_card", {"asset_id": A3, "campaign_id": C, "reason": "other"},
           expect={"skip": {"assetId": A3, "campaignId": C, "reason": "other", "source": "mcp"}})
check_tool("review_skip_card", {"asset_id": A3, "reason": "bored"}, "invalid_args", error_code="*")
check_tool("review_get_next", {"campaign_id": C}, "empty", expect="empty")

In [0]:
# Archive the run-scoped campaign (also the 409 case: an archived campaign cannot be reactivated).
if C != UNKNOWN:
    check_route("PATCH /api/campaigns/:campaignId/status", "happy_archive", params={"campaignId": C}, body={"status": "archived"},
                expect={"id": C, "status": "archived"})
    check_route("PATCH /api/campaigns/:campaignId/status", "error_409", params={"campaignId": C}, body={"status": "active"}, status=409)
    _c = check_route("GET /api/campaigns/:campaignId", "happy_after_cleanup", params={"campaignId": C}, expect={"status": "archived"})
    if not _c or _c.get("status") != "archived":  # the 409 check reactivated it: archive again, best effort
        http("PATCH", f"/api/campaigns/{C}/status", {"status": "archived"})
if MC:
    http("PATCH", f"/api/campaigns/{MC}/status", {"status": "archived"})
# The stewards group member was added and removed above; settings and notification preferences were restored.
# No API deletes assets, votes, skips, synonyms or examples: they stay, scoped by the unique TAG and domain ContractTest.
print(f"cleanup done; left in place: assets {sorted(SEED.values())} (tag {TAG})")

In [0]:
from pyspark.sql import types as T

# Every error status the contract lists gets a row: exercised live above, or skipped_live with the reason.
LIVE_SKIP_REASON = {
    401: "the Apps proxy rejects unauthenticated calls before they reach the app; covered in-process",
    403: "the M2M SPN is an app admin, so every role check passes; covered in-process",
    409: "conflict state cannot be produced safely live; covered in-process",
    500: "needs a failing data layer; covered in-process",
    502: "needs a failing upstream (warehouse / UC); covered in-process",
    400: "no invalid input reachable over HTTP for this route; covered in-process",
    404: "no unknown-id input for this route; covered in-process",
}
seen = {r["check_id"] for r in RESULTS}
for key, r in ROUTES.items():
    for status in r["errors"]:
        if f"route:{key}:error_{status}" not in seen:
            skip_live(key, f"error_{status}", LIVE_SKIP_REASON[status])

# Coverage: every manifest route, tool and resource has at least one row.
by_kind = {}
for r in RESULTS:
    by_kind.setdefault(r["kind"], set()).add(r["path_or_tool"])
missing = ([f"route {k}" for k in ROUTES if k not in by_kind.get("route", set())]
           + [f"tool {n}" for n in TOOLS if n not in by_kind.get("tool", set())]
           + [f"resource {u}" for u in RESOURCES if u not in by_kind.get("resource", set())])
total = len(ROUTES) + len(TOOLS) + len(RESOURCES)
coverage_pct = round(100.0 * (total - len(missing)) / total, 1)
record("coverage:manifest", "invariant", "", "manifest.json", {"coverage_pct": 100.0},
       {"coverage_pct": coverage_pct, "missing": missing}, "pass" if not missing else "fail")

schema = T.StructType([
    T.StructField("run_id", T.StringType()), T.StructField("check_id", T.StringType()),
    T.StructField("kind", T.StringType()), T.StructField("method", T.StringType()),
    T.StructField("path_or_tool", T.StringType()), T.StructField("expected", T.StringType()),
    T.StructField("actual", T.StringType()), T.StructField("status", T.StringType()),
    T.StructField("duration_ms", T.LongType()), T.StructField("ts", T.TimestampType()),
])
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RESULTS_TABLE} (
  run_id STRING COMMENT 'Job run id (or manual-<epoch>)',
  check_id STRING COMMENT '<kind>:<route key or tool>:<case>',
  kind STRING COMMENT 'route | tool | resource | invariant',
  method STRING, path_or_tool STRING, expected STRING, actual STRING,
  status STRING COMMENT 'pass | fail | skipped_live',
  duration_ms BIGINT, ts TIMESTAMP
) COMMENT 'Live API contract test results (ground-truth-app api_contract_tests job)'
""")
spark.createDataFrame(RESULTS, schema).write.mode("append").saveAsTable(RESULTS_TABLE)

counts = {s: sum(1 for r in RESULTS if r["status"] == s) for s in ("pass", "fail", "skipped_live")}
failed = [r["check_id"] for r in RESULTS if r["status"] == "fail"]
print(f"{len(RESULTS)} rows -> {RESULTS_TABLE}: {counts}; coverage {coverage_pct}%")
if failed or missing:
    raise AssertionError(f"{len(failed)} failed checks, coverage {coverage_pct}% (missing {missing}): {failed[:40]}")
dbutils.notebook.exit(json.dumps({"run_id": RUN_ID, **counts, "coverage_pct": coverage_pct}))